# Data Cleaning and Quote Quality

Raw option-chain data needs to be preserved first, then cleaned with explicit quote-quality rules. This notebook adds mid prices, bid-ask spreads, time to maturity, moneyness, log-moneyness, and exclusion flags.

## Quote-quality fields

| Field | Meaning |
|---|---|
| `mid_price` | Average of bid and ask |
| `bid_ask_spread` | Ask minus bid |
| `spread_pct` | Bid-ask spread divided by mid price |
| `time_to_maturity` | Years from snapshot time to expiry |
| `moneyness` | Underlying price divided by strike |
| `log_moneyness` | Natural log of strike divided by underlying price |
| `is_excluded` | True when a quote fails at least one quality filter |
| `exclusion_reason` | Primary reason a quote was removed |

In [ ]:
from pathlib import Path
import sys

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [ ]:
from src.cleaning import clean_option_chain, retained_quotes, save_cleaned_option_chain
from src.quote_quality import (
    data_quality_summary,
    exclusion_reason_summary,
    plot_data_cleaning_summary,
    plot_spread_by_expiry,
    plot_spread_by_moneyness,
    spread_by_expiry,
    spread_by_moneyness_bin,
)

## Load the latest raw option-chain snapshot

The cleaning step starts from a frozen CSV in `data/raw`. This prevents later analysis from changing every time the live option chain updates.

In [ ]:
raw_data_dir = project_root / "data" / "raw"
snapshot_files = sorted(raw_data_dir.glob("*option_chain_snapshot*.csv"))

if not snapshot_files:
    raise FileNotFoundError("No option-chain snapshot found in data/raw.")

raw_snapshot_path = snapshot_files[-1]
raw_options = pd.read_csv(raw_snapshot_path)

raw_snapshot_path

In [ ]:
raw_options.head(10)

## Underlying price

Moneyness requires the underlying price at the time of the snapshot. For live analysis, use the underlying price observed near the snapshot time. The value below should be updated to match the raw snapshot.

In [ ]:
underlying_price = 600.0

## Clean quotes and add flags

The cleaning rules flag missing bid/ask values, zero bids, crossed markets, wide spreads, expired contracts, invalid strikes, and low-liquidity contracts.

In [ ]:
cleaned_options = clean_option_chain(
    raw_options,
    underlying_price=underlying_price,
    max_spread_pct=0.40,
    min_volume=1,
    min_open_interest=1,
)

cleaned_options.head(10)

In [ ]:
cleaned_output_path = save_cleaned_option_chain(
    cleaned_options,
    output_dir=project_root / "data" / "processed",
    file_name="cleaned_option_chain.csv",
)

cleaned_output_path

## Data-quality summary

In [ ]:
quality_summary = data_quality_summary(cleaned_options)
quality_summary

In [ ]:
reason_summary = exclusion_reason_summary(cleaned_options)
reason_summary

## Retained quotes

The retained dataset is the starting point for implied-volatility and Greek calculations.

In [ ]:
clean_quotes = retained_quotes(cleaned_options)
clean_quotes.head(10)

## Spread by moneyness

A wider spread means the executable option price is less precise. This matters directly for implied-volatility uncertainty.

In [ ]:
spread_moneyness_summary = spread_by_moneyness_bin(cleaned_options, bins=10)
spread_moneyness_summary

## Spread by expiry

Shorter expiries can have different quote-quality behavior from longer expiries, especially around weekly contracts and less active strikes.

In [ ]:
spread_expiry_summary = spread_by_expiry(cleaned_options)
spread_expiry_summary

## Save figures

In [ ]:
figures_dir = project_root / "figures"
figures_dir.mkdir(exist_ok=True)

plot_data_cleaning_summary(
    cleaned_options,
    figures_dir / "data_cleaning_summary.png",
)

plot_spread_by_moneyness(
    cleaned_options,
    figures_dir / "bid_ask_spread_by_moneyness.png",
)

plot_spread_by_expiry(
    cleaned_options,
    figures_dir / "bid_ask_spread_by_expiry.png",
)

## Cleaning notes

The excluded rows are not deleted from the cleaned dataset. They remain in the file with flags and exclusion reasons, which makes the filtering process auditable.

Bad quotes can break implied-volatility solving because the solver tries to find the volatility that matches the observed option price. If the observed price is stale, crossed, zero, or too wide to be meaningful, the implied volatility can be unstable or impossible to solve.